# MnesOS Engine Showcase

**MnesOS** is a stateless, agentic RPG game engine. This notebook plays the role of the **client** — it owns and persists game state across turns while the engine itself remains stateless.

## Architecture recap

```
Client turn loop
  │
  ├─ append player message to client_messages
  ├─ call  app.invoke(state)  ──► stateless graph
  │           │
  │           ├─ Lore     : TF-IDF RAG over bot_lore.md
  │           ├─ Director : LLM maps intent → YARE events
  │           ├─ Tools    : YAREInterpreter executes deterministic rules
  │           ├─ NPC_Brain: LLM governs all NPCs
  │           ├─ Tools    : (NPC actions resolved)
  │           └─ Narrator : LLM writes final prose
  │
  └─ state returned ─ caller persists updated state for next turn
```

Every `app.invoke(state)` call is independent. The caller is responsible for threading state from one turn into the next.

## 1. Setup

Install / confirm dependencies, then import the engine components.  
Set `OPENAI_API_KEY` in your environment before running — the three LLM nodes (Director, NPC_Brain, Narrator) all use OpenAI.

In [ ]:
import sys, os, functools, pprint

# Ensure the src directory is importable when running from notebooks/
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(".")), "src"))

from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
from langchain_openai import ChatOpenAI

from MnesOS import (
    CartridgeLoader,
    GameState,
    trigger_event,
)
from MnesOS.graph import (
    reset_agent_messages_node,
    cleanup_agent_messages_node,
    context_retrieval_node,
    director_node,
    npc_brain_node,
    narrator_node,
    pre_tools_node,
    post_tools_node,
    route_director,
    route_npc_brain,
    route_rules,
)

print("MnesOS imported successfully.")


## 2. Load the Cartridge

A **cartridge** bundles three files:

| File | Purpose |
|---|---|
| `yare.yaml` | Deterministic state schema + procedural event rules |
| `bot_lore.md` | Markdown lore fed to TF-IDF RAG on every turn |
| `prompt_directives.yaml` | Narrative tone hints for the three LLM nodes |

`CartridgeLoader` validates all three at load time and returns a `LoadedCartridge` with `yare_config`, `prompt_directives`, `lore_path`, and a pre-populated `initial_state` derived from schema defaults.

In [ ]:
CARTRIDGE_DIR = os.path.join(os.path.dirname(os.path.abspath(".")), "cartridges", "generic-rpg")

loader = CartridgeLoader()
cartridge = loader.load(CARTRIDGE_DIR)

print("=== YARE events ===")
for name in cartridge.yare_config.get("events", {}):
    print(f"  • {name}")

print("\n=== Prompt directive keys ===")
for k in cartridge.prompt_directives:
    print(f"  • {k}")

print("\n=== Initial game state ===")
pprint.pprint(cartridge.initial_state)

## 3. Build the Graph with LLMs

The `workflow` exported from `MnesOS.graph` is a bare `StateGraph` whose LLM nodes default to `llm=None` (pass-through mode, useful for testing). To run a live game we wire real `ChatOpenAI` instances in via `functools.partial` and compile a fresh graph.

In [ ]:
# Three separate LLM instances — tune temperature per role.
# Director is deterministic (temperature=0): it maps intent → events.
# NPC_Brain is slightly creative (temperature=0.5): tactical NPC decisions.
# Narrator is the most expressive (temperature=0.8): immersive prose.
llm_director  = ChatOpenAI(model="gpt-4o-mini", temperature=0)
llm_npc_brain = ChatOpenAI(model="gpt-4o-mini", temperature=0.5)
llm_narrator  = ChatOpenAI(model="gpt-4o-mini", temperature=0.8)

# Rebuild the StateGraph, binding each LLM node via functools.partial.
graph = StateGraph(GameState)

graph.add_node("ResetAgentMessages", reset_agent_messages_node)
graph.add_node("Lore",               context_retrieval_node)
graph.add_node("Director",           functools.partial(director_node,  llm=llm_director))
graph.add_node("PreTools",           pre_tools_node)
graph.add_node("Tools",              ToolNode([trigger_event], messages_key="agent_messages"))
graph.add_node("PostTools",          post_tools_node)
graph.add_node("NPC_Brain",          functools.partial(npc_brain_node, llm=llm_npc_brain))
graph.add_node("Narrator",           functools.partial(narrator_node,  llm=llm_narrator))
graph.add_node("CleanupAgentMessages", cleanup_agent_messages_node)

graph.set_entry_point("ResetAgentMessages")
graph.add_edge("ResetAgentMessages", "Lore")
graph.add_edge("Lore", "Director")

graph.add_conditional_edges("Director",  route_director,  {"PreTools": "PreTools", "NPC_Brain": "NPC_Brain"})
graph.add_edge("PreTools", "Tools")
graph.add_edge("Tools",    "PostTools")
graph.add_conditional_edges("NPC_Brain", route_npc_brain, {"PreTools": "PreTools", "Narrator": "Narrator"})
graph.add_conditional_edges("PostTools", route_rules,     {"Director": "Director",  "NPC_Brain": "NPC_Brain"})

graph.add_edge("Narrator",             "CleanupAgentMessages")
graph.add_edge("CleanupAgentMessages", END)

app = graph.compile()
print("Graph compiled. Nodes:", list(app.get_graph().nodes.keys()))


## 4. Initialise Game State

`GameState` is a `TypedDict`. The caller owns this dict and threads it from turn to turn.

Key fields:

| Field | Owner | Description |
|---|---|---|
| `client_messages` | Caller | Full story history (`role`/`content` dicts). Appended to each turn. |
| `agent_messages` | Engine | Per-turn internal tool traffic. Always empty on entry/exit. |
| `bot_memory` | Engine | Deterministic world state updated by `YAREInterpreter`. |
| `system_notes` | Engine | Human-readable outcome notes from YARE rules. |

In [ ]:
state: GameState = {
    # Story history — the caller appends player messages before each invoke.
    "client_messages": [],
    # Agent-internal messages — always empty at turn boundaries.
    "agent_messages": [],
    # Deterministic world state seeded from yare.yaml defaults.
    "bot_memory": cartridge.initial_state,
    # Staging buffer for YARE state updates (cleared by PreTools, committed by PostTools).
    "bot_memory_staging": [],
    # Engine configuration (passed through every invoke unchanged).
    "yare_config":         cartridge.yare_config,
    "prompt_directives":   cartridge.prompt_directives,
    "lore_path":           cartridge.lore_path,
    # Cleared by Narrator at the end of each turn.
    "system_notes":   [],
    "retrieved_lore": "",
    "iteration_count": 0,
    "turn_phase":      "",
}

print("Initial bot_memory:")
pprint.pprint(state["bot_memory"])


## 5. Helper — play one turn

`play_turn` appends the player message, invokes the stateless graph once, and prints the narrator's response plus the updated world state.  The returned state becomes the input for the next turn — demonstrating the client's responsibility to persist state.

In [ ]:
def play_turn(current_state: GameState, player_message: str) -> GameState:
    """
    Play one turn of the game.

    The caller appends the player message, hands the state to the engine,
    then receives the updated state back.  State is the only communication
    channel — the graph itself is stateless.
    """
    # 1. The CLIENT appends the player's message to the story history.
    current_state["client_messages"].append({"role": "user", "content": player_message})

    print(f">>> Player: {player_message}\n")

    # 2. Invoke the stateless graph.  It returns the NEW state.
    new_state = app.invoke(current_state)

    # 3. Print the Narrator's response (last assistant message).
    narrator_response = next(
        (m["content"] for m in reversed(new_state["client_messages"]) if m["role"] == "assistant"),
        "(no narrator response)"
    )
    print(f"<<< Narrator:\n{narrator_response}\n")

    # 4. Print the deterministic world-state changes.
    print("─── World State ───────────────────────────────────────")
    pprint.pprint(new_state["bot_memory"])
    print("────────────────────────────────────────────────────────\n")

    # 5. Return the new state — the caller must persist this for the next turn.
    return new_state

## Turn 1 — Arrive at the Crossroads

The player's first message has no mechanical impact. The **Director** recognises it as a flavour action and calls no tools, so the graph routes directly to the Narrator for a purely descriptive arrival scene.

In [ ]:
state = play_turn(state, "I arrive at the Crossroads and look around. What do I see?")

## Turn 2 — Attack the Goblin

A physical attack. The **Director** calls `combat_strike(attacker="player", defender="npc", power=0)`. The **YAREInterpreter** rolls `1d20`, applies damage if ≥ 10, and records a system note. The **NPC_Brain** then decides whether the goblin retaliates.

In [ ]:
state = play_turn(state, "A goblin leaps from the bushes! I draw my sword and strike at it.")

## Turn 3 — Cast a Spell

A mana-consuming spell. The **Director** calls `cast_spell`. The interpreter checks `state.player.mana >= mana_cost` before deducting mana and dealing damage — pure deterministic logic with no LLM involved in the outcome.

In [ ]:
state = play_turn(state, "I raise my hand and cast Fireball at the goblin, spending 10 mana!")

## Turn 4 — Flee!

The **Director** calls the `flee` event. The interpreter rolls `1d20`; if ≥ 15 the player escapes and `current_location` is set to `"Safe Haven"`. This showcases the YARE `branch` + `set` actions altering world state deterministically.

In [ ]:
state = play_turn(state, "The goblin is relentless! I turn and run as fast as I can!")

## 6. Inspect the Full Story History

`client_messages` is the cumulative story transcript. The caller owns it and supplies it verbatim on every `invoke` call so the LLM nodes have full conversation context.

In [ ]:
print(f"Total messages in story history: {len(state['client_messages'])}\n")
for i, msg in enumerate(state["client_messages"], 1):
    role = msg["role"].upper()
    preview = msg["content"][:120].replace("\n", " ")
    print(f"[{i:02d}] {role:9s}  {preview}{'…' if len(msg['content']) > 120 else ''}")

## 7. Statelessness Proof

The graph has no memory of its own. Pass the same initial state twice with different player messages and you get two independent results — neither run affects the other.

In [ ]:
import copy

# Two independent clients, same starting world.
state_a = {
    "client_messages": [{"role": "user", "content": "I attack the goblin!"}],
    "agent_messages": [],
    "bot_memory":       copy.deepcopy(cartridge.initial_state),
    "bot_memory_staging": [],
    "yare_config":      cartridge.yare_config,
    "prompt_directives":cartridge.prompt_directives,
    "lore_path":        cartridge.lore_path,
    "system_notes": [],
    "retrieved_lore": "",
    "iteration_count": 0,
    "turn_phase": "",
}
state_b = copy.deepcopy(state_a)
state_b["client_messages"] = [{"role": "user", "content": "I try to run away!"}]

result_a = app.invoke(state_a)
result_b = app.invoke(state_b)

print("=== Run A: bot_memory after 'attack' ===")
pprint.pprint(result_a["bot_memory"])

print("\n=== Run B: bot_memory after 'run' ===")
pprint.pprint(result_b["bot_memory"])

print("\n>>> Run A and Run B are independent — the engine held no state between them.")


## Summary

| Concern | Where it lives |
|---|---|
| Game rules & dice | `yare.yaml` + `YAREInterpreter` — fully deterministic |
| World knowledge | `bot_lore.md` — TF-IDF RAG, no external API |
| Player intent → events | Director LLM |
| NPC tactics | NPC Brain LLM |
| Prose narration | Narrator LLM |
| Persistent state | **The caller** — passed in, returned, re-passed |

Swap the cartridge folder to load a completely different game without changing any engine code.